In [ ]:
!pip install diffusers transformers accelerate torch torchvision safetensors --quiet

# Importing libraries and defining helper function

In [ ]:
import torch
from diffusers import (
    StableDiffusionInstructPix2PixPipeline,
    StableDiffusionImg2ImgPipeline,
    StableDiffusionXLImg2ImgPipeline,
    StableDiffusionControlNetPipeline,
    ControlNetModel
)
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

In [3]:

def show_images(original, edited, title):
    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.imshow(original)
    plt.title("Original")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(edited)
    plt.title(title)
    plt.axis("off")

    plt.show()

def save_images(original, edited, model_name, prompt):
    os.makedirs("outputs", exist_ok=True)

    model_clean = model_name.replace("/", "_")
    prompt_clean = prompt.replace(" ", "_")

    original.save(f"outputs/{model_clean}_{prompt_clean}_original.png")
    edited.save(f"outputs/{model_clean}_{prompt_clean}_edited.png")

# Load Input Image

In [4]:
image_path = "scenery.jpg"
image = Image.open(image_path).convert("RGB")

prompt = "Make it a sunset with warm orange lighting"

# Model 1: InstructPix2Pix

In [ ]:
model_name = "timbrooks/instruct-pix2pix"

pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
    model_name,
    torch_dtype=dtype
).to(device)

edited = pipe(prompt=prompt, image=image).images[0]

show_images(image, edited, "InstructPix2Pix")
save_images(image, edited, model_name, prompt)

# Model 2: SDXL (Image-to-Image)

In [ ]:
model_name = "stabilityai/stable-diffusion-xl-base-1.0"

pipe = StableDiffusionXLImg2ImgPipeline.from_pretrained(
    model_name,
    torch_dtype=dtype
).to(device)

edited = pipe(
    prompt=prompt,
    image=image,
    strength=0.6,
    guidance_scale=7.5
).images[0]

show_images(image, edited, "SDXL")
save_images(image, edited, model_name, prompt)

# Model 3: Stable Diffusion v1.5

In [ ]:
model_name = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    model_name,
    torch_dtype=dtype
).to(device)

edited = pipe(
    prompt=prompt,
    image=image,
    strength=0.6,
    guidance_scale=7.5
).images[0]

show_images(image, edited, "SD v1.5")
save_images(image, edited, model_name, prompt)

# Model 4: ControlNet (Canny Edge + SD 1.5)

In [ ]:
model_name = "lllyasviel/sd-controlnet-canny"

controlnet = ControlNetModel.from_pretrained(model_name, torch_dtype=dtype)

pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype
).to(device)
/Users/riyagarg/Downloads/lllyasviel_sd-controlnet-canny_Make_it_a_sunset_with_warm_orange_lighting_original.png
image_np = np.array(image)
edges = cv2.Canny(image_np, 100, 200)
edges = np.stack([edges]*3, axis=2)
edge_image = Image.fromarray(edges)

edited = pipe(
    prompt=prompt,
    image=edge_image,
    guidance_scale=7.5
).images[0]

show_images(image, edited, "ControlNet Canny")
save_images(image, edited, model_name, prompt)